# Load and rehydrate the release

This notebook loads the five dehydrated tables, restores a matching edit pair from a full or partial WildChat source, and reconstructs the span IDs referenced by its actions. It ships without executed outputs or source text.

In [ ]:
from pathlib import Path
import sys

# Works whether the kernel starts in notebooks/, the release root, or the repo root.
ROOT = next(
    p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "public_release")
    if (p / "code" / "helpers.py").exists()
)
sys.path.insert(0, str(ROOT / "code"))

from helpers import (
    DATA_FILES,
    build_span_index,
    iter_table,
    read_table,
    rehydrate_from_wildchat,
    resolve_action_spans,
)

DATA_DIR = ROOT / "data"

## Inspect the dehydrated tables

The tables join through `prompt_id`, `tree_id`, and `pair_id`. Reading them as iterators avoids loading an entire file when only a few rows are needed.

In [ ]:
for name, filename in DATA_FILES.items():
    first_row = read_table(name, DATA_DIR, limit=1)[0]
    print(f"{filename}: {', '.join(first_row)}")

In [ ]:
candidate_limit = None
candidate_edges = []
for row in iter_table("edges", DATA_DIR):
    if row["edge_status"] == "in_scope":
        candidate_edges.append(row)
        if candidate_limit is not None and len(candidate_edges) == candidate_limit:
            break
candidate_prompt_ids = sorted({
    prompt_id
    for edge in candidate_edges
    for prompt_id in (edge["parent_prompt_id"], edge["child_prompt_id"])
})
len(candidate_edges), len(candidate_prompt_ids)

## Rehydrate available prompts

WildChat-4.8M-Full is gated. Accept its access terms and authenticate locally before running this cell. You may replace `DATASET_NAME` with a compatible subset. Missing IDs produce a warning and the matching rows are still returned. Released trees describe the full snapshot; recompute clustering, tree construction, and pruning before treating a subset as an independent topology.

In [ ]:
DATASET_NAME = "yuntian-deng/WildChat-4.8M-Full"
hydrated = rehydrate_from_wildchat(
    candidate_prompt_ids,
    dataset_name=DATASET_NAME,
    token=True,
    strict=False,
)
complete_edges = [
    edge
    for edge in candidate_edges
    if edge["parent_prompt_id"] in hydrated and edge["child_prompt_id"] in hydrated
]

{
    "requested_prompts": len(candidate_prompt_ids),
    "rehydrated_prompts": len(hydrated),
    "complete_candidate_edges": len(complete_edges),
}

## Reconstruct the action spans

Span IDs are assigned by an unrestricted, case-insensitive word-level `SequenceMatcher`. Character ranges are zero-based and half-open.

In [ ]:
if complete_edges:
    edge = complete_edges[0]
    tree = next(row for row in iter_table("trees", DATA_DIR) if row["tree_id"] == edge["tree_id"])
    nodes = [row for row in iter_table("nodes", DATA_DIR) if row["tree_id"] == edge["tree_id"]]
    actions = [row for row in iter_table("actions", DATA_DIR) if row["pair_id"] == edge["pair_id"]]
    expected_conversations = {row["prompt_id"]: row["conversation_id"] for row in nodes}
    assert all(
        hydrated[prompt_id]["conversation_id"] == expected_conversations[prompt_id]
        for prompt_id in (edge["parent_prompt_id"], edge["child_prompt_id"])
    )
    parent = hydrated[edge["parent_prompt_id"]]["prompt"]
    child = hydrated[edge["child_prompt_id"]]["prompt"]
    span_index = build_span_index(parent, child)
    reconstructed = [
        {"target": action["target"], "direction": action["direction"], **resolve_action_spans(action, span_index)}
        for action in actions
    ]
else:
    tree = edge = None
    nodes = actions = reconstructed = []
    print("No complete edit pair was found in this subset. Select prompt IDs present in your source.")

{"tree": tree, "edge": edge, "actions": reconstructed}